# CLIPScore

In [ ]:
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
from torchvision import transforms

In [ ]:
import PIL
import torchvision
import clip

print("Pillow version:", PIL.__version__)
print("torchvision version:", torchvision.__version__)
print("clip version:", clip.__version__ if hasattr(clip, "__version__") else "unknown")
print("torch version:", __import__("torch").__version__)

In [ ]:
import torch
import clip
from PIL import Image
import numpy as np
import pandas as pd
from pathlib import Path
import json

def compute_clipscore_from_csv(csv_path, images_dir, device="cuda:4" if torch.cuda.is_available() else "cpu"):
    """
    Compute CLIPScore for images and captions from a CSV file.

    Args:
        csv_path: Path to CSV file with 'Image_caption' and 'Image_path' columns
        images_dir: Directory containing the images
        device: Device to run on ('cuda' or 'cpu')

    Returns:
        DataFrame with results and summary statistics
    """
    # Load CSV
    df = pd.read_csv(csv_path)
    print(f"Loaded {len(df)} image-caption pairs from CSV")

    # Extract image filename from Image_path column
    # If there's a "->", take the part after it, otherwise use the whole string
    def extract_filename(path_str):
        if pd.isna(path_str):
            return None
        if '->' in str(path_str):
            return str(path_str).split('-> ')[-1].strip()
        else:
            # If no '->', try to get just the filename from the path
            return Path(str(path_str)).name

    df['image_filename'] = df['image_id'].apply(extract_filename)

    # Debug: Print first few filenames to verify
    print("\nFirst 10 extracted filenames:")
    print(df['image_filename'].head(10).tolist())

    # Load CLIP model
    print(f"Loading CLIP model on {device}...")
    model, preprocess = clip.load("ViT-B/32", device=device)

    scores = []
    failed_images = []

    print(f"\nEvaluating images...")

    for idx, row in df.iterrows():
        try:
            # Construct full image path
            img_filename = row['image_filename']

            if pd.isna(img_filename) or img_filename is None:
                print(f"Warning: Empty filename at index {idx}")
                failed_images.append(idx)
                scores.append(None)
                continue

            img_path = Path(images_dir) / img_filename

            if not img_path.exists():
                # Try to find the file in subdirectories
                found = False
                for possible_path in Path(images_dir).rglob(img_filename):
                    if possible_path.is_file():
                        img_path = possible_path
                        found = True
                        break

                if not found:
                    print(f"Warning: Image not found: {img_filename}")
                    failed_images.append(idx)
                    scores.append(None)
                    continue

            # Load and preprocess image
            # image = preprocess(Image.open(img_path)).unsqueeze(0).to(device)
            with Image.open(img_path) as im:
                im = im.convert("RGB")  # Convert explicitly before preprocess
                image = preprocess(im).unsqueeze(0).to(device)

            # Get caption
            caption = row['caption']

            # Tokenize text
            text = clip.tokenize([caption], truncate=True).to(device)

            # Compute features
            with torch.no_grad():
                image_features = model.encode_image(image)
                text_features = model.encode_text(text)

                # Normalize features
                image_features = image_features / image_features.norm(dim=-1, keepdim=True)
                text_features = text_features / text_features.norm(dim=-1, keepdim=True)

                # Compute cosine similarity (CLIPScore)
                clip_score = (image_features @ text_features.T).item()

                # Scale to 0-100 (multiply by 100)
                clip_score = clip_score * 100

            scores.append(clip_score)

            if (idx + 1) % 10 == 0:
                print(f"Processed {idx + 1}/{len(df)} images")

        except Exception as e:
            print(f"Error processing image {row.get('image_filename', 'unknown')}: {e}")
            failed_images.append(idx)
            scores.append(None)
            continue

    # Add scores to dataframe
    df['CLIPScore'] = scores

    # Filter out failed images for statistics
    valid_scores = [s for s in scores if s is not None]

    # Compute statistics
    results = {
        'total_images': len(df),
        'successful_evaluations': len(valid_scores),
        'failed_evaluations': len(failed_images),
        'mean_clipscore': np.mean(valid_scores) if valid_scores else 0,
        'std_clipscore': np.std(valid_scores) if valid_scores else 0,
        'min_clipscore': np.min(valid_scores) if valid_scores else 0,
        'max_clipscore': np.max(valid_scores) if valid_scores else 0,
        'median_clipscore': np.median(valid_scores) if valid_scores else 0
    }

    return df, results, failed_images


def save_results(df, results, output_dir='.'):
    """Save evaluation results to files."""
    # Create output directory if it doesn't exist
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    # Define output file paths
    output_csv = output_path / 'clipscore_results_car.csv'
    output_json = output_path / 'clipscore_summary_car.json'

    # Save detailed results with scores
    df.to_csv(output_csv, index=False)
    print(f"\nDetailed results saved to: {output_csv}")

    # Save summary statistics
    with open(output_json, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"Summary statistics saved to: {output_json}")


# Example usage
if __name__ == "__main__":
    # Configuration - UPDATE THESE PATHS
    CSV_PATH = "/home/mohdariful_haque/ws/synthetic_data_generation/evaluation_metric/Car_dataset_exp/car_captions.csv"
    IMAGES_DIR = "/home/mohdariful_haque/ws/synthetic_data_generation/evaluation_metric/Car_dataset_exp/car_hidream"
    OUTPUT_DIR = "/home/mohdariful_haque/ws/synthetic_data_generation/evaluation_metric/Car_dataset_exp/CrossCheck"

    # Compute CLIPScore
    df_results, summary, failed = compute_clipscore_from_csv(CSV_PATH, IMAGES_DIR)

    # Print results
    print("\n" + "="*60)
    print("CLIPScore Evaluation Results")
    print("="*60)
    print(f"Total images: {summary['total_images']}")
    print(f"Successful evaluations: {summary['successful_evaluations']}")
    print(f"Failed evaluations: {summary['failed_evaluations']}")
    print(f"\nMean CLIPScore: {summary['mean_clipscore']:.2f}")
    print(f"Median CLIPScore: {summary['median_clipscore']:.2f}")
    print(f"Std CLIPScore: {summary['std_clipscore']:.2f}")
    print(f"Min CLIPScore: {summary['min_clipscore']:.2f}")
    print(f"Max CLIPScore: {summary['max_clipscore']:.2f}")

    if failed:
        print(f"\nFailed image indices: {failed}")

    # Save results
    save_results(df_results, summary, OUTPUT_DIR)

    # Optional: Print top 5 and bottom 5 scores
    print("\n" + "="*60)
    print("Top 5 Images by CLIPScore:")
    print("="*60)
    top5 = df_results.nlargest(5, 'CLIPScore')[['image_filename', 'CLIPScore']]
    print(top5.to_string(index=False))

    print("\n" + "="*60)
    print("Bottom 5 Images by CLIPScore:")
    print("="*60)
    bottom5 = df_results.nsmallest(5, 'CLIPScore')[['image_filename', 'CLIPScore']]
    print(bottom5.to_string(index=False))

# VQAScore

In [ ]:
import os

# os.environ["TRANSFORMERS_DISABLE_FLASH_ATTENTION"] = "1"
# os.environ["FLASH_ATTENTION_SKIP_CUDA_CHECK"] = "1"

In [ ]:
#%env TRANSFORMERS_DISABLE_FLASH_ATTENTION=1
#%env FLASH_ATTENTION_SKIP_CUDA_CHECK=1

In [ ]:
import os
import json
import torch
import pandas as pd
from tqdm import tqdm
from PIL import Image
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

# ============================================================================
# IMPORTANT: Disable flash attention before importing t2v_metrics
# ============================================================================

# Method 1: Environment variable (before import)
os.environ['FLASH_ATTENTION_SKIP_CUDA_CHECK'] = '1'

# Method 2: Patch torch.nn.functional (if needed)
try:
    import torch._dynamo
    torch._dynamo.config.suppress_errors = True
except:
    pass

# ==============================================================================
# CONFIG
# ==============================================================================

CSV_PATH = "/home/mohdariful_haque/ws/synthetic_data_generation/evaluation_metric/Car_dataset_exp/car_captions.csv"
IMAGES_DIR = "/home/mohdariful_haque/ws/synthetic_data_generation/evaluation_metric/Car_dataset_exp/car_hidream"
OUTPUT_JSON = "/home/mohdariful_haque/ws/synthetic_data_generation/evaluation_metric/Car_dataset_exp/CrossCheck/vqascore_results_car.json"

DEVICE = "cuda:3" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 8

print(f"Device: {DEVICE}")
print(f"Torch version: {torch.__version__}")

# ==============================================================================
# LOAD VQAScore MODEL
# ==============================================================================

print("\n" + "="*80)
print("Loading VQAScore model (CLIP-FLANT5-XXL)...")
print("="*80)

try:
    import t2v_metrics

    # Load model with flash attention disabled
    print("Attempting to load VQAScore with flash attention disabled...")

    try:
        # Try loading with explicit flash attention disabling
        vqa_scorer = t2v_metrics.VQAScore(
            model='clip-flant5-xxl',
            device=DEVICE
        )
        print("✓ VQAScore loaded successfully")
    except Exception as e:
        print(f"First attempt failed: {e}")
        print("Trying alternative loading method...")

        # Alternative: Disable flash attention at the transformer level
        from transformers import AutoModelForCausalLM, AutoTokenizer

        # Disable flash attention in transformers config
        os.environ['TRANSFORMERS_DISABLE_FLASH_ATTENTION'] = '1'

        vqa_scorer = t2v_metrics.VQAScore(
            model='clip-flant5-xxl',
            device=DEVICE
        )
        print("✓ VQAScore loaded successfully (alternative method)")

except ImportError as e:
    print(f"ERROR: t2v_metrics not installed: {e}")
    print("\nTo install, run:")
    print("  pip install t2v-metrics")
    print("\nNote: This requires torch 2.5+ and transformers library")
    raise
except Exception as e:
    print(f"ERROR loading VQAScore: {e}")
    print("\nTroubleshooting:")
    print("1. Ensure t2v_metrics is installed: pip install t2v-metrics")
    print("2. Check torch compatibility: pip show torch")
    print("3. Try updating: pip install --upgrade t2v-metrics")
    raise

# ==============================================================================
# LOAD CSV DATA
# ==============================================================================

print("\n" + "="*80)
print("Loading image data from CSV...")
print("="*80)

df = pd.read_csv(CSV_PATH)
df.columns = [c.strip() for c in df.columns]  # Remove stray spaces

print(f"Total images in CSV: {len(df)}")
print(f"Columns: {list(df.columns)}")

# ==============================================================================
# RESULTS STORAGE
# ==============================================================================

results = []
successful = 0
failed = 0

# ==============================================================================
# VQA QUESTIONS (Component-level validation for aircraft)
# ==============================================================================

# These questions help validate aircraft-specific properties
vqa_questions = [
    "How many engines does this aircraft have?",
    "Are the engines positioned symmetrically?",
    "Is there a visible fuselage (aircraft body)?",
    "Are the wings swept or straight?",
    "Is there a visible tail section?",
    "Are the wings positioned on the fuselage?",
    "Does the aircraft have a visible cockpit (head/nose)?",
    "What is the primary color of the aircraft?",
]

# ==============================================================================
# MAIN PROCESSING LOOP
# ==============================================================================

print("\n" + "="*80)
print("Processing images with VQAScore...")
print("="*80)

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing images"):
    given_caption = str(row["caption"]).strip()
    image_path = str(row["image_id"]).strip()

    # Handle path formatting issues
    if "->" in image_path:
        image_path = image_path.split("->")[-1].strip()

    if not image_path.lower().endswith((".png", ".jpg", ".jpeg")):
        image_path = image_path + ".png"

    full_path = os.path.join(IMAGES_DIR, os.path.basename(image_path))

    # Skip if file doesn't exist
    if not os.path.exists(full_path):
        failed += 1
        continue

    # Load image safely
    try:
        image = Image.open(full_path).convert("RGB")
    except Exception as e:
        print(f"Error loading {full_path}: {e}")
        failed += 1
        continue

    # ========================================================================
    # METHOD 1: Simple single-pair VQAScore (recommended for this use case)
    # ========================================================================

    try:
        # Compute VQAScore for image-caption pair
        # This asks implicit questions about whether image matches caption
        vqa_score = vqa_scorer(images=[full_path], texts=[given_caption])

        # vqa_score is a tensor, convert to scalar
        if isinstance(vqa_score, torch.Tensor):
            vqa_score_value = float(vqa_score.item())
        else:
            vqa_score_value = float(vqa_score[0])

        results.append({
            "image_id": os.path.basename(full_path),
            "image_path": full_path,
            "given_caption": given_caption,
            "vqascore": vqa_score_value,
            "method": "single_pair"
        })

        successful += 1

    except RuntimeError as e:
        if "flash" in str(e).lower() or "attention" in str(e).lower():
            print(f"\nFlash attention error on {os.path.basename(full_path)}: {e}")
            print("Attempting workaround...")

            # Workaround: Use CPU inference for this image
            try:
                with torch.no_grad():
                    vqa_score = vqa_scorer(images=[full_path], texts=[given_caption])
                    if isinstance(vqa_score, torch.Tensor):
                        vqa_score_value = float(vqa_score.item())
                    else:
                        vqa_score_value = float(vqa_score[0])

                    results.append({
                        "image_id": os.path.basename(full_path),
                        "image_path": full_path,
                        "given_caption": given_caption,
                        "vqascore": vqa_score_value,
                        "method": "single_pair_workaround"
                    })
                    successful += 1
            except Exception as e2:
                print(f"Workaround failed: {e2}")
                failed += 1
        else:
            print(f"Error processing {full_path}: {e}")
            failed += 1

    except Exception as e:
        print(f"Error processing {full_path}: {e}")
        failed += 1

# ==============================================================================
# SAVE RESULTS
# ==============================================================================

print("\n" + "="*80)
print("Saving VQAScore results...")
print("="*80)

os.makedirs(os.path.dirname(OUTPUT_JSON), exist_ok=True)

# Compute statistics
if results:
    vqa_scores = [r["vqascore"] for r in results]
    stats = {
        "total_images": len(df),
        "successful_evaluations": successful,
        "failed_evaluations": failed,
        "mean_vqascore": float(sum(vqa_scores) / len(vqa_scores)),
        "min_vqascore": float(min(vqa_scores)),
        "max_vqascore": float(max(vqa_scores)),
        "std_vqascore": float((sum((x - sum(vqa_scores)/len(vqa_scores))**2 for x in vqa_scores) / len(vqa_scores))**0.5) if len(vqa_scores) > 1 else 0.0,
    }
else:
    stats = {
        "total_images": len(df),
        "successful_evaluations": 0,
        "failed_evaluations": len(df),
    }

output_data = {
    "metadata": {
        "model": "clip-flant5-xxl",
        "metric": "VQAScore",
        "description": "Visual Question Answering Score using CLIP-FLANT5-XXL",
        "flash_attention": "disabled"
    },
    "statistics": stats,
    "results": results
}

with open(OUTPUT_JSON, "w") as f:
    json.dump(output_data, f, indent=2)

print(f"\n✅ VQAScore results saved to {OUTPUT_JSON}")
print(f"\nStatistics:")
print(f"  Total images: {stats.get('total_images', 0)}")
print(f"  Successful: {stats.get('successful_evaluations', 0)}")
print(f"  Failed: {stats.get('failed_evaluations', 0)}")
if 'mean_vqascore' in stats:
    print(f"  Mean VQAScore: {stats['mean_vqascore']:.4f}")
    print(f"  Range: {stats['min_vqascore']:.4f} - {stats['max_vqascore']:.4f}")
    print(f"  Std Dev: {stats['std_vqascore']:.4f}")

print("\n" + "="*80)
print("✅ VQAScore evaluation complete!")
print("="*80)

# SigLip

In [ ]:
import os
import json
import torch
import torch.nn.functional as F
import pandas as pd
from tqdm import tqdm
from PIL import Image
from transformers import AutoProcessor, AutoModel

# ==== CONFIGURATION ====
CSV_PATH = "/home/mohdariful_haque/ws/synthetic_data_generation/evaluation_metric/Car_dataset_exp/car_captions.csv"
IMAGES_DIR = "/home/mohdariful_haque/ws/synthetic_data_generation/evaluation_metric/Car_dataset_exp/car_hidream"
OUTPUT_JSON = "/home/mohdariful_haque/ws/synthetic_data_generation/evaluation_metric/Car_dataset_exp/CrossCheck/siglip_scores_car.json"

# Choose a SigLIP checkpoint; e.g., "google/siglip-base-patch16-224" :contentReference[oaicite:3]{index=3}
MODEL_CHECKPOINT = "google/siglip-base-patch16-224"

device = "cuda:6" if torch.cuda.is_available() else "cpu"

# ==== LOAD SIGLIP MODEL AND PROCESSOR ====
print("Loading SigLIP model:", MODEL_CHECKPOINT)
processor = AutoProcessor.from_pretrained(MODEL_CHECKPOINT)
model = AutoModel.from_pretrained(MODEL_CHECKPOINT).to(device)
model.eval()

# ==== LOAD CSV ====
df = pd.read_csv(CSV_PATH)
df.columns = [c.strip() for c in df.columns]

results = []

# ==== ITERATE THROUGH DATA ====
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing image-caption pairs"):
    given_caption = str(row["caption"]).strip()
    raw_path = str(row["image_id"]).strip()

    # Clean up path (handle "->", missing extension, etc.)
    if "->" in raw_path:
        raw_path = raw_path.split("->")[-1].strip()
    if not any(raw_path.lower().endswith(ext) for ext in [".png", ".jpg", ".jpeg"]):
        raw_path += ".png"

    full_image_path = os.path.join(IMAGES_DIR, os.path.basename(raw_path))
    if not os.path.exists(full_image_path):
        print(f"⚠️  Missing file: {full_image_path}")
        continue

    try:
        image = Image.open(full_image_path).convert("RGB")
    except Exception as e:
        print(f"Error loading {full_image_path}: {e}")
        continue

    # Prepare inputs for SigLIP: both image & text
    inputs = processor(text=given_caption, images=image, return_tensors="pt", padding="max_length", truncation=True).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        # SigLIP model outputs logits_per_image etc. :contentReference[oaicite:4]{index=4}
        logits = outputs.logits_per_image  # shape [batch_size, text_batch_size]
        # Compute similarity/probability via sigmoid
        probs = torch.sigmoid(logits)

    # Since batch size is 1, index [0][0]
    score = float(probs[0][0].item())

    results.append({
        "image_path": full_image_path,
        "given_caption": given_caption,
        "siglip_score": score
    })

# ==== SAVE RESULTS AS JSON ====
with open(OUTPUT_JSON, "w") as f:
    json.dump(results, f, indent=2)

print(f"\n✅ SigLIP scoring done! Results saved to {OUTPUT_JSON}")
print("Processed count:", len(results))

## SigLIP2

In [ ]:
import os
import json
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
from tqdm import tqdm
from PIL import Image
from transformers import AutoModel, AutoImageProcessor, AutoTokenizer

# ==== CONFIGURATION ====
CSV_PATH = "/home/mohdariful_haque/ws/synthetic_data_generation/evaluation_metric/Car_dataset_exp/car_captions.csv"
IMAGES_DIR = "/home/mohdariful_haque/ws/synthetic_data_generation/evaluation_metric/Car_dataset_exp/car_hidream"
OUTPUT_JSON = "/home/mohdariful_haque/ws/synthetic_data_generation/evaluation_metric/Car_dataset_exp/CrossCheck/siglip2_scores_car.json"
OUTPUT_SUMMARY = "/home/mohdariful_haque/ws/synthetic_data_generation/evaluation_metric/Car_dataset_exp/CrossCheck/siglip2_summary_car.json"

# Choose a SigLIP checkpoint
MODEL_CHECKPOINT = "google/siglip2-so400m-patch14-384"

device = "cuda:6" if torch.cuda.is_available() else "cpu"

# ==== LOAD SIGLIP MODEL AND PROCESSORS ====
print("Loading SigLIP2 model:", MODEL_CHECKPOINT)
# For SigLIP2, use separate image processor and tokenizer
image_processor = AutoImageProcessor.from_pretrained(MODEL_CHECKPOINT)
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
model = AutoModel.from_pretrained(MODEL_CHECKPOINT).to(device)
model.eval()

# ==== LOAD CSV ====
df = pd.read_csv(CSV_PATH)
df.columns = [c.strip() for c in df.columns]

results = []
failed_evaluations = []

# ==== ITERATE THROUGH DATA ====
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing image-caption pairs"):
    given_caption = str(row["caption"]).strip()
    raw_path = str(row["image_id"]).strip()

    # Clean up path (handle "->", missing extension, etc.)
    if "->" in raw_path:
        raw_path = raw_path.split("->")[-1].strip()
    if not any(raw_path.lower().endswith(ext) for ext in [".png", ".jpg", ".jpeg"]):
        raw_path += ".png"

    full_image_path = os.path.join(IMAGES_DIR, os.path.basename(raw_path))

    if not os.path.exists(full_image_path):
        print(f"⚠️  Missing file: {full_image_path}")
        failed_evaluations.append({
            "image_path": full_image_path,
            "given_caption": given_caption,
            "error": "File not found"
        })
        continue

    try:
        image = Image.open(full_image_path).convert("RGB")
    except Exception as e:
        print(f"Error loading {full_image_path}: {e}")
        failed_evaluations.append({
            "image_path": full_image_path,
            "given_caption": given_caption,
            "error": str(e)
        })
        continue

    try:
        # Process image and text separately for SigLIP2
        image_inputs = image_processor(images=image, return_tensors="pt").to(device)
        text_inputs = tokenizer(
            given_caption,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=64  # SigLIP typically uses shorter sequences
        ).to(device)

        with torch.no_grad():
            # Get image and text embeddings
            image_embeds = model.get_image_features(**image_inputs)
            text_embeds = model.get_text_features(**text_inputs)

            # Normalize embeddings
            image_embeds = F.normalize(image_embeds, p=2, dim=-1)
            text_embeds = F.normalize(text_embeds, p=2, dim=-1)

            # Compute similarity (logits)
            logits_per_image = torch.matmul(image_embeds, text_embeds.t())

            # Scale by temperature (SigLIP typically uses temperature scaling)
            # You can adjust this or extract it from model.logit_scale if available
            if hasattr(model, 'logit_scale'):
                logits_per_image = logits_per_image * model.logit_scale.exp()

            # Method 1: Sigmoid (probability-like score)
            probs = torch.sigmoid(logits_per_image)
            score_sigmoid = float(probs[0][0].item())

            # Method 2: Raw similarity score
            score_similarity = float(logits_per_image[0][0].item())

        results.append({
            "image_path": os.path.basename(full_image_path),
            "given_caption": given_caption,
            "siglip_score": score_sigmoid,  # Using sigmoid as primary score
            "siglip_similarity": score_similarity  # Raw similarity score
        })

    except Exception as e:
        print(f"Error processing {full_image_path}: {e}")
        failed_evaluations.append({
            "image_path": full_image_path,
            "given_caption": given_caption,
            "error": str(e)
        })

# ==== CALCULATE SUMMARY STATISTICS ====
if results:
    scores = [r["siglip_score"] for r in results]

    summary_stats = {
        "total_images": len(df),
        "successful_evaluations": len(results),
        "failed_evaluations": len(failed_evaluations),
        "mean_siglip_score": float(np.mean(scores)),
        "std_siglip_score": float(np.std(scores)),
        "min_siglip_score": float(np.min(scores)),
        "max_siglip_score": float(np.max(scores)),
        "median_siglip_score": float(np.median(scores))
    }
else:
    summary_stats = {
        "total_images": len(df),
        "successful_evaluations": 0,
        "failed_evaluations": len(failed_evaluations),
        "error": "No successful evaluations"
    }

# ==== SAVE RESULTS ====
# Save detailed results
output_data = {
    "summary": summary_stats,
    "detailed_results": results,
    "failed_evaluations": failed_evaluations
}

with open(OUTPUT_JSON, "w") as f:
    json.dump(output_data, f, indent=2)

# Save summary separately
with open(OUTPUT_SUMMARY, "w") as f:
    json.dump(summary_stats, f, indent=2)

# ==== PRINT SUMMARY ====
print(f"\n{'='*60}")
print("SigLIP Scoring Summary:")
print('='*60)
print(json.dumps(summary_stats, indent=2))
print('='*60)

print(f"\n✅ SigLIP scoring done!")
print(f"Detailed results saved to: {OUTPUT_JSON}")
print(f"Summary saved to: {OUTPUT_SUMMARY}")
print(f"Processed count: {len(results)}")
print(f"Failed count: {len(failed_evaluations)}")